In [16]:
import numpy as np
import scipy.stats as stats

In [17]:
def InvPareto(x, th):
    return (1 - x)**(1 / (1 - th))

def med_func(th):
    return 2 ** (1/(th-1))

n = 100
beta = 0.95
theta = 10
Xn = InvPareto(stats.uniform(loc=0, scale=1).rvs(size=n), th=theta)
print(Xn)

[1.01128955 1.04438577 1.01913654 1.79288452 1.20266343 1.05373694
 1.00752585 1.04011137 1.1069176  1.1224128  1.07814194 1.08993947
 1.08338138 1.1159528  1.01050477 1.08526413 1.01495658 1.04778827
 1.01996994 1.03206778 1.05470744 1.09524588 1.15315298 1.04656633
 1.52167031 1.01522032 1.28930878 1.16295362 1.00271466 1.45515006
 1.04102621 1.07417174 1.02712913 1.02449196 1.07497024 1.12889638
 1.19999833 1.04053868 1.01505408 1.29808449 1.15131277 1.09559162
 1.14969606 1.12767105 1.21397362 1.13889062 1.07370796 1.13436943
 1.08897535 1.00456549 1.02338694 1.40003862 1.14063669 1.01128048
 1.0263929  1.0711642  1.3643237  1.03460301 1.16085651 1.08017247
 1.47549214 1.05415505 1.02811736 1.06494424 1.00488736 1.06889488
 1.11118423 1.47184498 1.09098009 1.02325172 1.17263649 1.02796068
 1.06693838 1.12676249 1.04976273 1.05857617 1.06446084 1.188882
 1.04453395 1.0720849  1.65052181 1.33726715 1.00090134 1.00319536
 1.05083673 1.01358682 1.11085521 1.21914147 1.13107541 1.098937

Асимптотический доверительный интервал для медианы

In [18]:
theta_omp = 1 + 1 / (np.mean(np.log(Xn)))
med = med_func(theta)
med_omp = med_func(theta_omp)
t1 = stats.norm.ppf((1-beta)/2)
t2 = stats.norm.ppf((1+beta)/2)
asymp_med_left = med_omp - med_omp*np.log(2)/(np.sqrt(n)*(theta_omp-1))*t2
asymp_med_right = med_omp - med_omp*np.log(2)/(np.sqrt(n)*(theta_omp-1))*t1
asymp_med_l = asymp_med_right - asymp_med_left
print(f"Медиана - {med}\nОценка медианы:{med_omp}")
print("Асимптотический доверительный интервал для медианы: ", asymp_med_left, "< theta <", asymp_med_right)
print("l= ", asymp_med_l)

Медиана - 1.080059738892306
Оценка медианы:1.0809193408149314
Асимптотический доверительный интервал для медианы:  1.064434395405075 < theta < 1.0974042862247877
l=  0.03296989081971269


Асимптотический доверительный интервал для theta

In [19]:
t1 = stats.norm.ppf((1-beta)/2)
t2 = stats.norm.ppf((1+beta)/2)
asymp_left = theta_omp - t2 / (np.sum(np.log(Xn))/np.sqrt(n))
asymp_right = theta_omp - t1 / (np.sum(np.log(Xn))/np.sqrt(n))
asymp_l = asymp_right - asymp_left
print(f"theta - {theta}, oценка theta ОМП:{theta_omp}")
print("Асимптотический доверительный интервал для theta:", asymp_left, "< theta <", asymp_right)
print("l =", asymp_l)

theta - 10, oценка theta ОМП:9.907981911138748
Асимптотический доверительный интервал для theta: 8.162049539062124 < theta < 11.653914283215371
l = 3.491864744153247


Непараметрический, параметрический бутстрапы

In [20]:
N_np = 1000

N_p = 50000

def bootstrap_p_OMP(x, N):
    arr = []
    n = len(x)
    for _ in range(N):
        sample = InvPareto(stats.uniform(loc=0, scale=1).rvs(size=n), th=theta_omp)
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]
    return sorted(np.array(arr))

def bootstrap_np_OMP(x, N):
    arr = []
    n = len(x)
    for _ in range(N):
        sample = np.random.choice(x, size=n, replace=True)
        arr += [(1 + 1 / (np.mean(np.log(sample)))) - theta]
    return sorted(np.array(arr))


bs_arr_np = bootstrap_np_OMP(Xn, N_np)
bs_arr_p = bootstrap_np_OMP(Xn, N_p)


t1_np = int((1 - beta)/2 * N_np - 1)
t2_np = int((1 + beta)/2 * N_np - 1)

t1_p = int((1 - beta)/2 * N_p - 1)
t2_p = int((1 + beta)/2 * N_p - 1)


bs_np_left = theta_omp - bs_arr_np[t2_np]
bs_np_right = theta_omp - bs_arr_np[t1_np]

bs_p_left = theta_omp - bs_arr_p[t2_p]
bs_p_right = theta_omp - bs_arr_p[t1_p]


bs_np_l = bs_np_right - bs_np_left

bs_p_l = bs_p_right - bs_p_left

print("Непараметрический бутстраповский доверительный интервал для theta(ОМП):", bs_np_left, "< theta <", bs_np_right)
print("NP_len = ", bs_np_l)

print("Параметрический бутстраповский доверительный интервал для theta(ОМП):", bs_p_left, "< theta <", bs_p_right)
print("P_len = ", bs_p_l)

Непараметрический бутстраповский доверительный интервал для theta(ОМП): 7.963512749255157 < theta < 11.613136338360484
NP_len =  3.6496235891053264
Параметрический бутстраповский доверительный интервал для theta(ОМП): 7.965626696345852 < theta < 11.501992063589316
P_len =  3.536365367243464


 Сравнение полученных доверительных интервалов

In [21]:
sort_lens = sorted([(asymp_l, "Асимптотический"), (bs_np_l, "Бутстрап непараметрический"), (bs_p_l, "Бутстрап параметрический")])
print("Рейтинг доверительных интервалов (для theta):")
for i in range(len(sort_lens)):
    print(f"{i+1})", sort_lens[i][1], f"(l = {np.round(sort_lens[i][0],3)})")

Рейтинг доверительных интервалов (для theta):
1) Асимптотический (l = 3.492)
2) Бутстрап параметрический (l = 3.536)
3) Бутстрап непараметрический (l = 3.65)
